# 08 - Validation and Interpretation of the Cluster-Based Recommender

## Role of This Notebook
This notebook does not train anything new. Its function is to validate, interpret, and stress-test the selected model together with the top 5 recommendation layer. It also leaves a clear guide for explaining how the system can later be used in Grasshopper with new simulation values.

## Questions It Answers
1. Which model was finally selected and where it is stored.
2. How reliable its predictions are over the `bridge clusters`.
3. Which variables seem to push the model decision the most.
4. How a cluster prediction is transformed into a top 5 species list.
5. How to reuse the pipeline with new simulated tiles in Grasshopper.


In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#777777',
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
EXPORT_DIR = ROOT / 'exports'

MODEL_PATH = MODELS_DIR / 'plant_cluster_classifier.joblib'
PREDICTIONS_PATH = PROCESSED_DIR / 'geometry_predictions.csv'
RECOMMENDATIONS_PATH = PROCESSED_DIR / 'tile_plant_recommendations_intermediate.csv'
PLANTS_PATH = PROCESSED_DIR / 'plants_encoded.csv'
CLUSTER_PROFILES_PATH = PROCESSED_DIR / 'plants_cluster_profiles.csv'
COMPARISON_PATH = PROCESSED_DIR / 'model_comparison.csv'
NOTES_PATH = PROCESSED_DIR / 'model_selection_notes.csv'
CONFUSION_PATH = PROCESSED_DIR / 'best_model_confusion_matrix.csv'
FINAL_EXPORT_PATH = EXPORT_DIR / 'tile_plant_recommendations.csv'

model = joblib.load(MODEL_PATH)
predictions = pd.read_csv(PREDICTIONS_PATH)
recommendations = pd.read_csv(RECOMMENDATIONS_PATH)
plants = pd.read_csv(PLANTS_PATH)
cluster_profiles = pd.read_csv(CLUSTER_PROFILES_PATH)
model_comparison = pd.read_csv(COMPARISON_PATH)
model_notes = pd.read_csv(NOTES_PATH, index_col=0)
confusion_matrix_df = pd.read_csv(CONFUSION_PATH, index_col=0)

predictions.head()

## 1. Artifact Inventory
Before interpreting results, it is useful to make clear which file is produced by each part of the pipeline. This table also serves as traceability evidence for the final delivery.


In [ ]:
artifacts = pd.DataFrame([
    {'artifact': 'plant_cluster_classifier.joblib', 'path': str(MODEL_PATH), 'role': 'Final supervised model for predicting bridge clusters from geometric features.'},
    {'artifact': 'geometry_predictions.csv', 'path': str(PREDICTIONS_PATH), 'role': 'Model predictions by row with cluster probabilities.'},
    {'artifact': 'tile_plant_recommendations_intermediate.csv', 'path': str(RECOMMENDATIONS_PATH), 'role': 'Top 5 species by row before the final export.'},
    {'artifact': 'plants_encoded.csv', 'path': str(PLANTS_PATH), 'role': 'Plant catalog already encoded, clustered, and ready for recommendation.'},
    {'artifact': 'plants_cluster_profiles.csv', 'path': str(CLUSTER_PROFILES_PATH), 'role': 'Perfiles promedio por cluster botanico y bridge cluster.'},
    {'artifact': 'tile_plant_recommendations.csv', 'path': str(FINAL_EXPORT_PATH), 'role': 'Final artifact for external query or integration.'}
])
display(artifacts)
display(model_notes)

## 2. KEY - Why the Pipeline Goes From KMeans to RandomForest
A key question in the defense is why the project uses `KMeans` first and then `RandomForest`, instead of choosing only one. The answer is that they solve different problems within a methodological sequence, according to what needs to be solved for future new locations.

### Role of KMeans
- works on the plant dataset
- does not need previous labels
- discovers groups of species with similar environmental requirements
- reduces the botanical catalog to operational profiles

### Role of RandomForest
- works on the geometry dataset
- does need a supervised label
- learns to predict which `bridge cluster` fits a new cell
- allows the system to be deployed with simulated tiles without recalculating the whole botanical crosswalk from scratch

### Methodological Conclusion
`KMeans` organizes the botanical universe. `RandomForest` makes that organization useful for prediction. That is why they do not replace each other: they are chained.

### Why Bridge Clusters Appear
The botanical clusters from notebook 04 are not always distinguishable from geometry. For that reason, several botanical clusters are consolidated into fewer `bridge clusters`, which are the groups actually predictable with the available spatial variables. This consolidation makes the pipeline more stable and more accurate for deployment.


## 3. General Reading of the Selected Model
This section makes visible which model was selected, which classes it predicts, and how it performed against its competitors. This is important because the final recommendation depends on the quality of this intermediate classification stage.


In [ ]:
classifier = model.named_steps['model']
model_summary = pd.Series({
    'pipeline_type': type(model).__name__,
    'estimator_type': type(classifier).__name__,
    'selected_model_note': model_notes.loc['selected_model', 'value'],
    'n_training_classes': len(getattr(classifier, 'classes_', [])),
    'training_classes': ', '.join(map(str, getattr(classifier, 'classes_', [])))
})
display(model_summary.to_frame('value'))
display(model_comparison.sort_values(['macro_f1', 'accuracy'], ascending=False))

### Conclusion About Model Selection
The comparison confirms that `random_forest` was not chosen for convenience, but because of numeric evidence. It ranks first with `accuracy = 0.999768` and `macro_f1 = 0.999201`, ahead of `logistic_regression`, `svm`, and `knn`. This reinforces the idea that the final model combines very strong fit with a practically perfect class balance.


## 4. Prediction Distribution and Confidence
The following section helps answer two practical questions: how many cases fall into each bridge cluster and with what average confidence those predictions are being issued.


In [ ]:
cluster_counts = predictions['plant_cluster_pred_label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(cluster_counts.index, cluster_counts.values, color='#4e79a7', edgecolor='#444444', linewidth=0.6)
axes[0].set_title('Predicted bridge cluster distribution')
axes[0].set_xlabel('Predicted bridge cluster')
axes[0].set_ylabel('Number of rows')
axes[0].tick_params(axis='x', rotation=45)

axes[1].hist(predictions['plant_cluster_pred_probability'].dropna(), bins=25, color='#59a14f', edgecolor='#444444', linewidth=0.6, alpha=0.85)
axes[1].set_title('Distribution of prediction confidence')
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### Conclusion About Distribution and Confidence
The distribution confirms that `bridge_cluster_2` dominates the prediction set, consistent with what was already observed in the bridge label. However, the classifier's average confidence is extremely high, close to `0.9974`, so the correct reading is not that the model is highly uncertain, but that the learned structure stabilized quite clearly on the current data.


## 5. Confusion Matrix and Error Interpretation
The confusion matrix is not only useful for measuring accuracy; it also helps explain where the model confuses nearby profiles and where it should not be confused.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
cm_values = confusion_matrix_df.to_numpy()
im = ax.imshow(cm_values, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='Count')
ax.set_xticks(range(len(confusion_matrix_df.columns)))
ax.set_yticks(range(len(confusion_matrix_df.index)))
ax.set_xticklabels(confusion_matrix_df.columns, rotation=45, ha='right')
ax.set_yticklabels(confusion_matrix_df.index)
for i in range(cm_values.shape[0]):
    for j in range(cm_values.shape[1]):
        ax.text(j, i, str(cm_values[i, j]), ha='center', va='center', color='black', fontsize=8)
ax.set_title('Confusion matrix of the selected classifier')
plt.tight_layout()
plt.show()
display(confusion_matrix_df)

### Conclusion About Model Errors
The confusion matrix shows that the system practically does not make mistakes in the evaluated set. This conclusion is important not only because of the number of correct predictions, but also because it greatly reduces the risk that a cell ends up associated with a botanically distant bridge cluster. As a result, the top 5 recommendation inherits a very stable supervised base.


## 6. Variables With the Greatest Weight in the Model
Because the selected model was a `RandomForest`, we can inspect its feature importance. This does not replace causal interpretation, but it does help support the idea that the model is reading plausible patterns from the dataset.


In [ ]:
feature_columns = ['BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W', 'UDI', 'SUN_HOURS', 'RADIATION']
if hasattr(classifier, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': feature_columns,
        'importance': classifier.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(8, 4.5))
    plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='#f28e2b', edgecolor='#444444', linewidth=0.6)
    plt.title('Feature importance in the selected RandomForest')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    display(importance_df)
else:
    print('The selected model does not expose feature_importances_.')

### Conclusion About Feature Importances
The `RandomForest` importances should not be read as botanical causality, but as operational evidence that certain simulation variables weigh more when separating bridge clusters. In the defense, this makes it possible to argue that the model does not operate as a total black box: there are geometric and light variables that clearly organize the decision.


## 7. Example Cases for Explaining Results
A good defense does not rely only on global averages. It is also useful to show a few concrete cases: high-confidence cells and more ambiguous cells.


In [ ]:
analysis_df = predictions.merge(
    recommendations[['ROW_ID', 'top1_species', 'top1_score', 'top2_species', 'top2_score', 'top3_species', 'top3_score']],
    on='ROW_ID',
    how='left'
)

high_confidence_cases = analysis_df.sort_values('plant_cluster_pred_probability', ascending=False).head(3)
low_margin_cases = analysis_df.sort_values('bridge_margin', ascending=True).head(3)

print('High confidence cases:')
display(high_confidence_cases[['ROW_ID', 'TILE_ID', 'spatial_label', 'plant_cluster_pred_label', 'plant_cluster_pred_probability', 'top1_species', 'top1_score', 'top2_species', 'top2_score']])

print('Ambiguous cases by low bridge margin:')
display(low_margin_cases[['ROW_ID', 'TILE_ID', 'spatial_label', 'plant_cluster_pred_label', 'bridge_margin', 'top1_species', 'top1_score', 'top2_species', 'top2_score', 'top3_species', 'top3_score']])

## 8. Inference Functions for New Tiles
This section condenses the logic needed to use the model with new rows. It is useful both for internal tests and for a future Grasshopper integration.


In [ ]:
bridge_key_cols = ['sun_min_h_day', 'sun_max_h_day', 'lux_min', 'lux_max', 'sun_center_h_day', 'lux_center', 'plant_light_index']
if not {'bridge_cluster_id', 'bridge_cluster_label'}.issubset(plants.columns):
    if {'bridge_cluster_id', 'bridge_cluster_label'}.issubset(cluster_profiles.columns):
        bridge_map = cluster_profiles[['plant_variety_cluster', 'bridge_cluster_id', 'bridge_cluster_label']].drop_duplicates()
    else:
        bridge_profiles = cluster_profiles[bridge_key_cols].drop_duplicates().reset_index(drop=True).copy()
        bridge_profiles['bridge_cluster_id'] = bridge_profiles.index.astype(int)
        bridge_profiles['bridge_cluster_label'] = 'bridge_cluster_' + bridge_profiles['bridge_cluster_id'].astype(str)
        bridge_map = cluster_profiles[['plant_variety_cluster'] + bridge_key_cols].merge(bridge_profiles, on=bridge_key_cols, how='left')[['plant_variety_cluster', 'bridge_cluster_id', 'bridge_cluster_label']].drop_duplicates()
    plants = plants.merge(bridge_map, on='plant_variety_cluster', how='left')

feature_columns = ['BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W', 'UDI', 'SUN_HOURS', 'RADIATION']

def minmax_against_reference(value, reference_series):
    min_val = reference_series.min()
    max_val = reference_series.max()
    if max_val == min_val:
        return 0.0
    return (value - min_val) / (max_val - min_val)

def build_model_row(tile_dict):
    row = dict(tile_dict)
    if row['SUN_HOURS'] > 24:
        row['sun_h_day_estimated'] = row['SUN_HOURS'] / 365.0
    else:
        row['sun_h_day_estimated'] = row['SUN_HOURS']

    row['sun_norm'] = minmax_against_reference(row['SUN_HOURS'], predictions['SUN_HOURS'])
    row['radiation_norm'] = minmax_against_reference(row['RADIATION'], predictions['RADIATION'])
    row['udi_norm'] = minmax_against_reference(row['UDI'], predictions['UDI'])
    row['tile_light_index'] = 0.45 * row['sun_norm'] + 0.45 * row['radiation_norm'] + 0.10 * row['udi_norm']

    return pd.DataFrame([{col: row[col] for col in feature_columns + ['sun_h_day_estimated', 'tile_light_index']}])

def extract_cluster_id_from_probability_column(col):
    value = col.replace('pred_cluster_', '').replace('_probability', '')
    if value.startswith('bridge_'):
        value = value.replace('bridge_', '')
    return int(value)

def predict_cluster_and_top5(tile_dict):
    prepared = build_model_row(tile_dict)
    X_new = prepared[feature_columns]
    pred_label = model.predict(X_new)[0]
    pred_probability = float(model.predict_proba(X_new).max())

    classifier = model.named_steps['model']
    class_labels = [str(c) for c in classifier.classes_]
    class_probabilities = model.predict_proba(X_new)[0]
    prob_lookup = {}
    for label, prob in zip(class_labels, class_probabilities):
        if label.startswith('bridge_cluster_'):
            cluster_id = int(label.replace('bridge_cluster_', ''))
        elif label.startswith('cluster_'):
            cluster_id = int(label.replace('cluster_', ''))
        else:
            cluster_id = int(label)
        prob_lookup[cluster_id] = prob

    plant_bridge_ids = plants['bridge_cluster_id'].astype(int).to_numpy()
    cluster_affinity = np.array([prob_lookup.get(c, 0.0) for c in plant_bridge_ids], dtype=float)
    light_fit = np.exp(-(((prepared.iloc[0]['tile_light_index'] - plants['plant_light_index'].fillna(plants['plant_light_index'].median()).to_numpy(dtype=float)) / 0.25) ** 2))
    indoor_bonus = np.minimum(plants['indoor_use_score'].fillna(0).to_numpy(dtype=float) * 5, 15)
    flex_bonus = np.minimum(plants['light_flexibility'].fillna(0).to_numpy(dtype=float) * 3, 10)
    temp_bonus = np.minimum(plants['temp_context_score'].fillna(0).to_numpy(dtype=float) * 4, 8)
    score = (cluster_affinity * 70) + (light_fit * 15) + indoor_bonus + flex_bonus + temp_bonus

    top_indices = np.argsort(score)[-5:][::-1]
    top5 = plants.iloc[top_indices][['latin', 'common', 'plant_cluster_label', 'bridge_cluster_label']].copy()
    top5['recommendation_score'] = score[top_indices]

    return {
        'prepared_row': prepared,
        'predicted_cluster_label': pred_label,
        'predicted_probability': pred_probability,
        'top5': top5.reset_index(drop=True)
    }

## 9. Test With a New Cell
Here, a new tile is simulated with input values compatible with the project. This test shows that the model does not depend on the full historical dataset to generate a recommendation.


In [ ]:
new_tile_example = {
    'BUILDING_ORIENTATION': 0,
    'RATIO_N': 0.15,
    'RATIO_E': 0.35,
    'RATIO_S': 0.30,
    'RATIO_W': 0.20,
    'UDI': 62.0,
    'SUN_HOURS': 510.0,
    'RADIATION': 138.0
}

new_tile_result = predict_cluster_and_top5(new_tile_example)
print('Predicted cluster:', new_tile_result['predicted_cluster_label'])
print('Predicted probability:', round(new_tile_result['predicted_probability'], 4))
display(new_tile_result['prepared_row'])
display(new_tile_result['top5'])

## 10. How to Bring This to Grasshopper
The Grasshopper implementation can be done with a `GhPython` component or with an external call to Python. The central idea is always the same:

1. receive the same model inputs per cell
2. build the row with the same preprocessing used in notebook 05
3. load `plant_cluster_classifier.joblib`
4. predict the `bridge cluster` and its probabilities
5. apply the same ranking rule from notebook 06
6. return the top 5 species, scores, and cluster

### Minimum Inputs Grasshopper Must Provide per Tile
- `BUILDING_ORIENTATION`
- `RATIO_N`
- `RATIO_E`
- `RATIO_S`
- `RATIO_W`
- `UDI`
- `SUN_HOURS`
- `RADIATION`

### Artifacts Grasshopper Must Be Able to Read
- `models/plant_cluster_classifier.joblib`
- `data/processed/plants_encoded.csv`
- `data/processed/plants_cluster_profiles.csv`

### Operational Pseudocode for GhPython
```python
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

ROOT = Path(r'C:\path	o\AyudaDavidAgudelo')
model = joblib.load(ROOT / 'models' / 'plant_cluster_classifier.joblib')
plants = pd.read_csv(ROOT / 'data' / 'processed' / 'plants_encoded.csv')
cluster_profiles = pd.read_csv(ROOT / 'data' / 'processed' / 'plants_cluster_profiles.csv')
reference_predictions = pd.read_csv(ROOT / 'data' / 'processed' / 'geometry_predictions.csv')

def minmax_against_reference(value, series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return 0.0
    return (value - min_val) / (max_val - min_val)

def prepare_tile(tile):
    sun_h_day_estimated = tile['SUN_HOURS'] / 365.0 if tile['SUN_HOURS'] > 24 else tile['SUN_HOURS']
    sun_norm = minmax_against_reference(tile['SUN_HOURS'], reference_predictions['SUN_HOURS'])
    radiation_norm = minmax_against_reference(tile['RADIATION'], reference_predictions['RADIATION'])
    udi_norm = minmax_against_reference(tile['UDI'], reference_predictions['UDI'])
    tile_light_index = 0.45 * sun_norm + 0.45 * radiation_norm + 0.10 * udi_norm
    row = pd.DataFrame([{
        'BUILDING_ORIENTATION': tile['BUILDING_ORIENTATION'],
        'RATIO_N': tile['RATIO_N'],
        'RATIO_E': tile['RATIO_E'],
        'RATIO_S': tile['RATIO_S'],
        'RATIO_W': tile['RATIO_W'],
        'UDI': tile['UDI'],
        'SUN_HOURS': tile['SUN_HOURS'],
        'RADIATION': tile['RADIATION'],
        'sun_h_day_estimated': sun_h_day_estimated,
        'tile_light_index': tile_light_index
    }])
    return row

tile = {
    'BUILDING_ORIENTATION': BUILDING_ORIENTATION,
    'RATIO_N': RATIO_N,
    'RATIO_E': RATIO_E,
    'RATIO_S': RATIO_S,
    'RATIO_W': RATIO_W,
    'UDI': UDI,
    'SUN_HOURS': SUN_HOURS,
    'RADIATION': RADIATION
}

prepared = prepare_tile(tile)
X_new = prepared[['BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W', 'UDI', 'SUN_HOURS', 'RADIATION']]
pred_label = model.predict(X_new)[0]
pred_proba = model.predict_proba(X_new)[0]

# The final Grasshopper output can be
```


## 11. KEY - Two Script Levels for Grasshopper
Two complementary versions remain in the `scripts/` folder:

1. `grasshopper_plant_recommender.py`
   - more complete version
   - organized into functions and artifacts
   - recommended for robust tests or external integration

2. `grasshopper_plant_recommender_minimal.py`
   - more direct version
   - intended as a base to copy or adapt inside `GhPython`
   - ideal when the important thing is quickly connecting tile inputs with the top 5 species

The methodological recommendation is to use the complete version first to validate the flow and then, if Grasshopper deployment requires it, move to the minimalist version.


## 12. KEY - Notes on Using the Minimalist Script
The minimalist version is intended to be used as follows:

### Expected Inputs per Tile
- `BUILDING_ORIENTATION`
- `RATIO_N`
- `RATIO_E`
- `RATIO_S`
- `RATIO_W`
- `UDI`
- `SUN_HOURS`
- `RADIATION`

### Expected Output
The script returns a dictionary with:
- `predicted_cluster_label`
- `predicted_probability`
- `tile_light_index`
- `top5` as a species table with score

### Recommended Use in GhPython
1. Place the `.py` script in an accessible path.
2. Add that path to `sys.path`.
3. Import `recommend_for_tile`.
4. Pass the tile values as direct arguments.
5. Split the output into lists for visualization or export.

### Conceptual Call Example
```python
import sys
sys.path.append(r'C:\path	o\project\scripts')
from grasshopper_plant_recommender_minimal import recommend_for_tile

result = recommend_for_tile(
    BUILDING_ORIENTATION,
    RATIO_N,
    RATIO_E,
    RATIO_S,
    RATIO_W,
    UDI,
    SUN_HOURS,
    RADIATION
)

pred_cluster = result['predicted_cluster_label']
pred_prob = result['predicted_probability']
top5_table = result['top5']
```

### Conclusion for Practical Use
If `predicted_probability` is high, the suggested cluster can be read as a strong recommendation. If it is medium or low, the top 5 should be read with more openness, because several species or even several bridge clusters may still be plausible. With the project's current results, the average probability is very high, so Grasshopper use can be framed with considerable confidence for initial recommendation tests.
